<a href="https://colab.research.google.com/github/AdvayRoongta/audio-initial-dataset/blob/main/scripts/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiments: audio frames -> LLM tokens

How much of Qwen2-Audio's input does one second of audio actually take up?

The audio encoder downsamples in fixed steps, so clip duration maps to a
predictable number of token positions. This notebook measures that mapping,
which we need before any patching / attention work -- otherwise we don't know
*which* positions in the sequence are the audio.

Sections 1-4 only need the **processor** (no GPU, no 17 GB download).
Section 5 loads the full model and needs a GPU.

Run `load_datasets.ipynb` first so the clips exist in Drive.

## Setup

In [ ]:
import os

# Get to the repo root: clone it on a fresh Colab
if os.path.exists("requirements.txt"):
    pass
elif os.path.exists("../requirements.txt"):
    %cd ..
else:
    !git clone -q https://github.com/AdvayRoongta/audio-initial-dataset.git repo
    %cd repo

!pip install -q -r requirements.txt


## Point at the cached clips

Same Drive folder `load_datasets.ipynb` wrote to. Nothing is downloaded here.

In [ ]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd()

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/audio-initial-dataset")
except ImportError:
    DATA_ROOT = REPO_ROOT

with open("examples.json") as f:
    examples = json.load(f)

# Paths in examples.json are relative to DATA_ROOT.
missing = [e["id"] for e in examples if not (DATA_ROOT / e["audio_path"]).exists()]
print(f"{len(examples)} examples, {len(missing)} missing audio")
if missing:
    print("run load_datasets.ipynb first ->", missing[:5])


## 1. The downsampling chain

Audio goes through fixed-size steps before the language model sees it:

```
waveform          16 kHz
mel spectrogram, hop=160            -> 100 frames/sec
conv stride 2                       ->  50 frames/sec
average pool stride 2               ->  25 tokens/sec
audio tokens      25/sec = 40 ms each, then projected to the LM's width
```

**Duration sets the number of token positions**.

In [ ]:
from transformers import AutoProcessor, Qwen2AudioConfig

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
config = Qwen2AudioConfig.from_pretrained(MODEL_ID)

fe = processor.feature_extractor
SR = fe.sampling_rate
FRAMES_PER_SEC = SR / fe.hop_length          # mel frames per second
TOKENS_PER_SEC = FRAMES_PER_SEC / 4          # two stride-2 steps
SEC_PER_TOKEN = 1 / TOKENS_PER_SEC

print(f"sampling rate     {SR}")
print(f"mel frames/sec    {FRAMES_PER_SEC:.0f}")
print(f"audio tokens/sec  {TOKENS_PER_SEC:.0f}   ({SEC_PER_TOKEN*1000:.0f} ms per token)")
print(f"audio encoder dim {config.audio_config.d_model}")
print(f"LM hidden dim     {config.text_config.hidden_size}")
print(f"audio token id    {config.audio_token_index}")


## 2. Duration -> number of tokens

The two stride-2 steps use integer arithmetic, so the exact count is:

```
n = ((mel_frames - 1) // 2 + 1 - 2) // 2 + 1
```

`n_audio_tokens` is what we compare against the processor's real output.

In [ ]:
def n_mel_frames(duration_sec):
    """Mel frames the feature extractor produces for a clip this long."""
    return int(duration_sec * FRAMES_PER_SEC)


def n_audio_tokens(mel_frames):
    """Mel frames -> audio token positions (conv stride 2, then pool stride 2)."""
    after_conv = (mel_frames - 1) // 2 + 1
    return (after_conv - 2) // 2 + 1


for d in [1, 2.5, 5, 10, 30]:
    print(f"{d:>5} s -> {n_mel_frames(d):>5} mel frames -> {n_audio_tokens(n_mel_frames(d)):>4} tokens")


## 3. Check it against the real processor

The feature extractor always pads to 30 s, so `input_features` is the same
shape for every clip. The number of *audio tokens* isn't because it comes from
`feature_attention_mask`. We must confirm homogeneity.

In [ ]:
import inspect

import librosa

AUDIO_TOKEN_ID = config.audio_token_index

# transformers renamed this kwarg (audios= -> audio=) and silently ignores the
# wrong one, so pick whichever this version actually takes.
AUDIO_KWARG = "audio" if "audio" in inspect.signature(processor.__call__).parameters else "audios"


def encode(example):
    """Run one example through the processor. Returns inputs + duration."""
    path = DATA_ROOT / example["audio_path"]
    audio, _ = librosa.load(path, sr=SR)

    conversation = [{"role": "user", "content": [
        {"type": "audio", "audio_url": str(path)},
        {"type": "text", "text": example["text_prompt"]},
    ]}]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=[text], sampling_rate=SR, return_tensors="pt",
                       **{AUDIO_KWARG: [audio]})
    return inputs, len(audio) / SR


ex = examples[0]
inputs, duration = encode(ex)

valid_frames = int(inputs["feature_attention_mask"].sum())
actual = int((inputs["input_ids"] == AUDIO_TOKEN_ID).sum())
predicted = n_audio_tokens(valid_frames)

print(f"clip                 {ex['id']}  ({duration:.2f} s)")
print(f"input_features       {tuple(inputs['input_features'].shape)}   <- padded to 30 s")
print(f"valid mel frames     {valid_frames}")
print(f"audio tokens         {actual}  (formula predicted {predicted})")
print(f"total prompt tokens  {inputs['input_ids'].shape[-1]}")
assert actual == predicted, "formula disagrees with the processor"


Now sweep every example to confirm the relationship is linear.

In [ ]:
rows = []
for e in examples:
    if not (DATA_ROOT / e["audio_path"]).exists():
        continue
    inputs, duration = encode(e)
    rows.append({
        "id": e["id"],
        "conflict_type": e["conflict_type"],
        "duration_sec": round(duration, 2),
        "audio_tokens": int((inputs["input_ids"] == AUDIO_TOKEN_ID).sum()),
        "total_tokens": inputs["input_ids"].shape[-1],
    })

for r in rows[:10]:
    print(f"{r['id']:<16} {r['duration_sec']:>5.2f} s  {r['audio_tokens']:>4} audio  "
          f"{r['total_tokens']:>4} total  ({r['audio_tokens']/r['duration_sec']:.1f} tok/s)")
print(f"\n{len(rows)} examples encoded")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.scatter([r["duration_sec"] for r in rows], [r["audio_tokens"] for r in rows], s=18)
plt.xlabel("clip duration (s)")
plt.ylabel("audio tokens")
plt.title(f"~{TOKENS_PER_SEC:.0f} audio tokens per second")
plt.grid(alpha=0.3)
plt.show()


## 4. Which token covers which moment

Each audio token is one 40 ms slice, in order. This allows us to map the token index to the timestamp and vice versa.

In [ ]:
def token_to_time(k):
    """Audio token k -> (start, end) in seconds."""
    return k / TOKENS_PER_SEC, (k + 1) / TOKENS_PER_SEC


def time_to_token(t):
    """Timestamp in seconds -> audio token index."""
    return int(t * TOKENS_PER_SEC + 1e-9)  # epsilon: floats land just under bin edges


def audio_token_positions(inputs):
    """Where the audio sits in the prompt: (first, last) index into input_ids."""
    pos = (inputs["input_ids"][0] == AUDIO_TOKEN_ID).nonzero().flatten()
    return int(pos[0]), int(pos[-1])


inputs, duration = encode(examples[0])
start, end = audio_token_positions(inputs)

print(f"audio occupies input_ids[{start}:{end+1}]  ({end - start + 1} positions)")
print(f"token 0   covers {token_to_time(0)[0]:.2f}-{token_to_time(0)[1]:.2f} s")
print(f"token 10  covers {token_to_time(10)[0]:.2f}-{token_to_time(10)[1]:.2f} s")
mid = duration / 2
print(f"time {mid:.2f}s falls in audio token {time_to_token(mid)} "
      f"-> input_ids position {start + time_to_token(mid)}")


This mapping is positional, not a claim
about receptive fields. The encoder runs 32 layers of *global* self-attention
before pooling, so every token has already seen the whole clip. This step isn't the part we are investigating.

## 5. Confirm against the real model (GPU)

Everything above is processor-only. This last section loads the 7B model to
check the actual tensor shapes coming out of the audio encoder.

In [ ]:
import torch
from transformers import Qwen2AudioForConditionalGeneration

# eager is required: sdpa returns an EMPTY attentions tuple (no error), and
# section 6 needs those weights.
load_kwargs = dict(device_map="auto", attn_implementation="eager")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if vram_gb < 20:
    # 8.4B params in fp16 = 16.8 GB, which won't fit a 15 GB T4.
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    print(f"{vram_gb:.0f} GB VRAM -> loading in 4-bit")
else:
    load_kwargs["torch_dtype"] = torch.float16

model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print("loaded:", model.dtype, model.device)


In [ ]:
inputs, duration = encode(examples[0])
inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

# Newer transformers moved the encoder under model.model.
audio_tower = getattr(model, "audio_tower", None) or model.model.audio_tower

# Cross-check our formula against the model's own length function.
valid_frames = int(inputs["feature_attention_mask"].sum())
_, official = audio_tower._get_feat_extract_output_lengths(torch.tensor([valid_frames]))
print(f"tokens: ours={n_audio_tokens(valid_frames)}  model's own fn={int(official[0])}")

# Audio encoder output, before it's projected into the LM.
with torch.no_grad():
    enc = audio_tower(inputs["input_features"].to(model.dtype)).last_hidden_state

print(f"\nclip duration        {duration:.2f} s")
print(f"encoder output       {tuple(enc.shape)}   (batch, frames, d_model)")
print(f"after pooling        ~{enc.shape[1] // 2} tokens -> projected to {config.text_config.hidden_size}")

# Hidden states at every layer: the LM's width is constant, length is what varies.
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)

print(f"\nLM layers            {len(out.hidden_states)}")
print(f"hidden state shape   {tuple(out.hidden_states[-1].shape)}   (batch, seq, hidden)")


## 6. Where does the model look? (attention mass)

Now the actual question: when the model answers, how much of its attention
goes to the audio versus the text prompt?

Two things to be careful about, both of which will skew the numbers if ignored:

- **Attention sinks.** The first few positions soak up attention as a no-op
  ([StreamingLLM](https://arxiv.org/abs/2309.17453)). We drop them from the
  denominator, otherwise we're mostly measuring how idle a head is.
- **Length.** The audio span is longer than the text span, so raw mass looks
  audio-heavy for free. We divide by the span's share of the sequence, so
  **1.0 = attends to audio exactly as much as its size warrants**, and
  < 1.0 = under-attends to audio.

In [ ]:
import math

import torch

N_SINK = 4  # first few positions act as attention sinks


def forward_with_attention(example):
    """Forward pass keeping attention weights. Returns (inputs, output, duration)."""
    inputs, duration = encode(example)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out = model(**inputs, output_attentions=True, use_cache=False)
    return inputs, out, duration


def audio_attention_mass(attentions, span, answer_pos=-1):
    """Per (layer, head): share of the answer position's attention landing on audio.

    Returns a (n_layers, n_heads) tensor. Sinks excluded from the total.
    """
    start, end = span
    n_layers, n_heads = len(attentions), attentions[0].shape[1]
    aam = torch.zeros(n_layers, n_heads)
    for l, A in enumerate(attentions):
        row = A[0, :, answer_pos, :].float().clone()   # (heads, keys)
        row[:, :N_SINK] = 0
        aam[l] = row[:, start:end + 1].sum(-1) / row.sum(-1).clamp(min=1e-9)
    return aam


def length_normalize(aam, span, seq_len):
    """1.0 = attention proportional to the span's size; <1 = under-attending."""
    share = (span[1] - span[0] + 1) / (seq_len - N_SINK)
    return aam / share


In [ ]:
ex = examples[0]
inputs, out, duration = forward_with_attention(ex)

span = audio_token_positions(inputs)
seq_len = inputs["input_ids"].shape[-1]

aam = audio_attention_mass(out.attentions, span)
aam_n = length_normalize(aam, span, seq_len)

print(f"{ex['id']}  ({ex['conflict_type']}, {duration:.2f}s)")
print(f"audio span {span}, {span[1]-span[0]+1} of {seq_len} tokens\n")
print(f"raw audio attention mass  {aam.mean():.3f}")
print(f"length-normalized         {aam_n.mean():.3f}   (1.0 = proportional)\n")

print("per layer (length-normalized):")
for l in range(aam_n.shape[0]):
    bar = "#" * int(aam_n[l].mean() * 20)
    print(f"  layer {l:>2}  {aam_n[l].mean():.2f}  {bar}")


In [ ]:
# Which individual heads attend to audio the most?
flat = aam_n.flatten()
top = flat.topk(10)

print("top heads by length-normalized audio attention:")
for score, idx in zip(top.values, top.indices):
    l, h = divmod(int(idx), aam_n.shape[1])
    print(f"  layer {l:>2} head {h:>2}   {float(score):.2f}")

TOP_LAYER, TOP_HEAD = divmod(int(top.indices[0]), aam_n.shape[1])
print(f"\nmost audio-focused head: layer {TOP_LAYER}, head {TOP_HEAD}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.imshow(aam_n.numpy(), aspect="auto", cmap="magma", origin="lower")
plt.colorbar(label="audio attention (length-normalized)")
plt.xlabel("head"); plt.ylabel("layer")
plt.title(f"Where {ex['id']} attends to audio")
plt.show()


## 7. Where *inside* the utterance?

Same attention row, but restricted to the audio span and renormalized into a
distribution over time. Normalized entropy tells us the shape:

- **near 1.0** = spread evenly over the clip -> a genuinely utterance-level trait
- **near 0** = concentrated on a few 40 ms moments

The three conflict types should differ here. Speaker identity is roughly
stationary (any moment will do), emotional prosody is a contour, and sarcasm
often lives in specific timing. If all three look alike, that itself says the
model isn't differentiating them.

In [ ]:
def attention_profile(attentions, span, layer, head, answer_pos=-1):
    """Attention over the audio span only, renormalized to sum to 1."""
    p = attentions[layer][0, head, answer_pos, span[0]:span[1] + 1].float()
    return p / p.sum().clamp(min=1e-9)


def normalized_entropy(p):
    """0 = all mass on one token, 1 = perfectly uniform."""
    q = p[p > 0]
    return float(-(q * q.log()).sum() / math.log(len(p)))


p = attention_profile(out.attentions, span, TOP_LAYER, TOP_HEAD)
peak = int(p.argmax())

print(f"layer {TOP_LAYER} head {TOP_HEAD}")
print(f"normalized entropy  {normalized_entropy(p):.3f}")
print(f"peak at audio token {peak} = {token_to_time(peak)[0]:.2f}-{token_to_time(peak)[1]:.2f}s"
      f"  of a {duration:.2f}s clip")


In [ ]:
times = [token_to_time(k)[0] for k in range(len(p))]

plt.figure(figsize=(9, 3))
plt.fill_between(times, p.numpy(), alpha=0.7)
plt.xlabel("time in clip (s)"); plt.ylabel("attention")
plt.title(f"{ex['id']}: layer {TOP_LAYER} head {TOP_HEAD} over the utterance")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
# Aggregate across every example, grouped by conflict type.
from collections import defaultdict

by_type = defaultdict(list)
for e in examples:
    if not (DATA_ROOT / e["audio_path"]).exists():
        continue
    inp, o, _ = forward_with_attention(e)
    sp = audio_token_positions(inp)
    a = length_normalize(audio_attention_mass(o.attentions, sp), sp, inp["input_ids"].shape[-1])
    by_type[e["conflict_type"]].append({
        "aam": float(a.mean()),
        "entropy": normalized_entropy(attention_profile(o.attentions, sp, TOP_LAYER, TOP_HEAD)),
    })
    del o  # attention tensors are large

print(f"{'conflict type':<20} {'n':>3}  {'audio attn':>10}  {'entropy':>8}")
for t, rs in by_type.items():
    n = len(rs)
    print(f"{t:<20} {n:>3}  {sum(r['aam'] for r in rs)/n:>10.3f}  "
          f"{sum(r['entropy'] for r in rs)/n:>8.3f}")


## 8. Does that attention actually matter? (knockout)

Attention maps are a hypothesis, not evidence — a head can attend to audio and
still not use it ([Jain & Wallace](https://arxiv.org/abs/1902.10186)). The test
is causal: block the answer position from attending to the audio span and see
whether the answer moves ([Geva et al.](https://arxiv.org/html/2304.14767)).

We pass a custom 4D additive mask, which replaces the causal mask entirely — so
we rebuild the causal part ourselves and add the knockout on top. The
`causal-only` run is the control: it must reproduce the baseline exactly, which
proves the mask is right before we trust any knockout number.

In [ ]:
def build_mask(seq_len, dtype, device, block_span=None, answer_pos=-1):
    """Causal mask, optionally blocking the answer position from a span."""
    neg = torch.finfo(dtype).min
    m = torch.zeros(1, 1, seq_len, seq_len, dtype=dtype, device=device)
    m.masked_fill_(torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device), 1), neg)
    if block_span is not None:
        m[0, 0, answer_pos, block_span[0]:block_span[1] + 1] = neg
    return m


def run_with_mask(inputs, mask):
    ins = {k: v for k, v in inputs.items() if k != "attention_mask"}
    with torch.no_grad():
        return model(**ins, attention_mask=mask, use_cache=False)


def option_logprobs(logits, example):
    """Log-prob of the audio-truth vs text-claim answer as the next token."""
    lp = torch.log_softmax(logits[0, -1].float(), -1)
    out = {}
    for key in ("ground_truth_audio", "ground_truth_text"):
        tid = processor.tokenizer.encode(" " + example[key], add_special_tokens=False)[0]
        out[key] = float(lp[tid])
    return out


In [ ]:
inputs, base_out, _ = forward_with_attention(ex)
span = audio_token_positions(inputs)
seq_len = inputs["input_ids"].shape[-1]
dtype, device = model.dtype, model.device

# Control: causal-only mask must match the baseline.
ctrl = run_with_mask(inputs, build_mask(seq_len, dtype, device))
same = torch.allclose(base_out.logits.float(), ctrl.logits.float(), atol=1e-3)
print(f"causal-only control reproduces baseline: {same}")
assert same, "mask construction is wrong -- knockout numbers would be meaningless"

# Knockout: answer position cannot see the audio.
ko = run_with_mask(inputs, build_mask(seq_len, dtype, device, block_span=span))

b = option_logprobs(base_out.logits, ex)
k = option_logprobs(ko.logits, ex)
pref_b = b["ground_truth_audio"] - b["ground_truth_text"]
pref_k = k["ground_truth_audio"] - k["ground_truth_text"]

print(f"\n{ex['id']}: audio={ex['ground_truth_audio']!r} text={ex['ground_truth_text']!r}\n")
print(f"{'':<10} {'audio':>8} {'text':>8} {'audio-text':>11}")
print(f"{'baseline':<10} {b['ground_truth_audio']:>8.2f} {b['ground_truth_text']:>8.2f} {pref_b:>11.2f}")
print(f"{'knockout':<10} {k['ground_truth_audio']:>8.2f} {k['ground_truth_text']:>8.2f} {pref_k:>11.2f}")
print(f"\nshift: {pref_k - pref_b:+.2f}")
print("negative = blocking audio pushed it toward the text answer, i.e. the")
print("attention was load-bearing. ~0 = the model wasn't using the audio anyway.")


### Scaling this up

The single-example run above is the scaffold. To make it a result, sweep every
example and report the mean shift per conflict type, plus the flip rate (how
often knockout changes which answer wins). Per-head knockout needs module hooks
rather than the 4D mask, since one mask applies to all layers — worth doing only
for the heads section 6 flagged as high-attention.

## Caveats

- On a T4 this runs in **4-bit**, which perturbs activations. Fine for locating
  *which* heads and *when*, but re-check any headline number in fp16.
- Attention is measured at the last prompt position, i.e. where the first answer
  token is decided. Attention during longer generations can differ.
- Raw per-layer attention understates mixing across layers; if the layer story
  matters, cross-check with
  [attention rollout](https://arxiv.org/abs/2005.00928).
- The audio encoder sees a 30 s padded input, so encoder-side analysis (not done
  here) must mask to the valid frames first.

## Next

We now know where the audio lives in the sequence and what each position
corresponds to in time. That's the addressing we need for attention analysis
and activation patching on the conflict examples.